# Phase 2 Kaggle Inference Notebook

Notebook này dùng để chạy inference HoVer-Net trên Kaggle và lưu đủ output cần cho Phase 2 feature extraction.

## Output sẽ gồm gì?
- `overlay/*.png`: ảnh overlay để kiểm tra trực quan
- `json/*.json`: metadata từng nucleus (`bbox`, `centroid`, `contour`, `type`, `type_prob`)
- `mat/*.mat`: file quan trọng nhất cho Phase 2, chứa `inst_map`, `inst_uid`, `inst_type`, `inst_centroid`
- `mat/*.mat` có thể chứa thêm `raw_map` nếu bật `--save_raw_map`

## Cần lưu gì để trích xuất đặc trưng?
- Nếu muốn trích xuất features sau này mà không inference lại, hãy giữ `mat/*.mat`.
- `json/*.json` chỉ đủ cho metadata nucleus, nhưng không đủ cho tất cả feature nếu thiếu `inst_map`.

## 1) Chuẩn bị đường dẫn

Sửa `REPO_URL`, `INPUT_DIR`, `MODEL_PATH` theo môi trường Kaggle của bạn.
- `REPO_URL`: source repo bạn muốn clone
- `INPUT_DIR`: thư mục ảnh input trên Kaggle
- `MODEL_PATH`: checkpoint HoVer-Net/PointNu-Net

In [ ]:
from pathlib import Path

REPO_URL = 'https://github.com/<your-account>/<your-repo>.git'
REPO_DIR = Path('/kaggle/working/hover_net')
INPUT_DIR = Path('/kaggle/input/<your-input-images-folder>')
OUTPUT_DIR = Path('/kaggle/working/phase2_outputs')
MODEL_PATH = REPO_DIR / 'checkpoints' / '<your-checkpoint>.pt'
TYPE_INFO_PATH = REPO_DIR / 'type_info.json'

print('REPO_DIR:', REPO_DIR)
print('INPUT_DIR:', INPUT_DIR)
print('OUTPUT_DIR:', OUTPUT_DIR)
print('MODEL_PATH:', MODEL_PATH)

In [ ]:
import os
import subprocess
from pathlib import Path

if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repo already exists:', REPO_DIR)

os.chdir(REPO_DIR)
print('Current working directory:', Path.cwd())

In [ ]:
# Kaggle thường đã có sẵn hầu hết dependency cần cho inference.
# Không cài nguyên requirements.txt vì file đó pin matplotlib==3.3.0 và các bản cũ khác, dễ lỗi trên Python mới.
import importlib
import subprocess
import sys

required_packages = [
    'docopt==0.6.2',
    'future==0.18.2',
    'imgaug==0.4.0',
]

for spec in required_packages:
    pkg_name = spec.split('==')[0]
    try:
        importlib.import_module(pkg_name)
        print(f'{pkg_name} already installed')
    except ImportError:
        print(f'Installing {spec}...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', spec], check=True)

print('Dependency setup complete')

## 2) Chạy inference CLI

Pipeline chính của `hover_net` sẽ tự làm:
1. đọc ảnh input
2. chạy model inference
3. post-process ra `inst_map` + `inst_info_dict`
4. lưu ảnh overlay, `.json`, `.mat`

Các tham số quan trọng:
- `--save_raw_map`: lưu raw prediction map vào `.mat` để debug
- `--save_qupath`: lưu thêm file `.tsv` cho QuPath
- `--draw_dot`: vẽ centroid nuclei trên overlay

Đối với Phase 2, file quan trọng nhất là `mat/*.mat` vì nó giữ `inst_map` và metadata để trích xuất đặc trưng sau này.

In [ ]:
import subprocess
import sys

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, 'run_infer.py',
    f'--model_path={MODEL_PATH}',
    '--model_mode=fast',
    '--nr_types=6',
    f'--type_info_path={TYPE_INFO_PATH}',
    '--nr_inference_workers=4',
    '--nr_post_proc_workers=2',
    '--batch_size=8',
    'tile',
    f'--input_dir={INPUT_DIR}',
    f'--output_dir={OUTPUT_DIR}',
    '--draw_dot',
    '--save_raw_map',
]

print('Running:', ' '.join(map(str, cmd)))
subprocess.run(cmd, check=True)

## 3) Output sau inference

Sau khi chạy xong, output thường nằm ở:
- `phase2_outputs/overlay/`
- `phase2_outputs/json/`
- `phase2_outputs/mat/`

### Dùng cái gì cho bước trích xuất đặc trưng?
- Dùng `mat/*.mat` là tối ưu nhất.
- Cần giữ: `inst_map`, `inst_type`, `inst_centroid`.
- Nếu muốn kiểm tra thủ công, dùng thêm `json/*.json` và `overlay/*.png`.
- `raw_map` chỉ cần khi debug model hoặc muốn xem lại đầu ra thô của inference.

In [ ]:
# Ví dụ đọc file .mat để chuẩn bị cho feature extraction
from scipy.io import loadmat
import numpy as np

mat_files = sorted((OUTPUT_DIR / 'mat').glob('*.mat'))
print('Number of mat files:', len(mat_files))

if mat_files:
    sample = loadmat(mat_files[0])
    print('Keys:', [k for k in sample.keys() if not k.startswith('__')])
    inst_map = sample['inst_map']
    inst_uid = sample.get('inst_uid')
    inst_type = sample.get('inst_type')
    inst_centroid = sample.get('inst_centroid')

    print('inst_map shape:', inst_map.shape)
    if inst_uid is not None:
        print('inst_uid shape:', inst_uid.shape)
    if inst_type is not None:
        print('inst_type shape:', inst_type.shape)
    if inst_centroid is not None:
        print('inst_centroid shape:', inst_centroid.shape)

    nuclei_count = int(np.max(inst_map))
    print('nuclei_count from inst_map:', nuclei_count)

## 4) Ghi nhớ cho Phase 2

- Inference output cuối cùng không phải là ảnh duy nhất.
- Kết quả thật sự cần cho Phase 2 là `inst_map` + metadata nuclei.
- Nếu chỉ lưu ảnh overlay thì không đủ để trích xuất đặc trưng sau này.
- Vì vậy hãy giữ `mat/*.mat` làm output chính, `json/*.json` làm backup, `overlay/*.png` để kiểm tra.